# Applio Darwin — Notebook de production

Ce notebook est conçu pour un runtime Colab/Kaggle jetable : le dépôt GitHub contient le code, Google Drive conserve les fichiers lourds et le runtime est reconstruit automatiquement à chaque session.

Le dépôt est déjà configuré pour `LizibaMvuluzi/applio-darwin`. Aucune URL GitHub n'est à modifier dans le notebook.

**Authentification GitHub :** le dépôt étant privé, la cellule de récupération utilise le secret Colab `GITHUB_TOKEN` s'il est disponible. Ce secret est stocké dans Colab, jamais dans le dépôt.


## 1. Récupérer automatiquement le projet depuis GitHub

In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO = "https://github.com/LizibaMvuluzi/applio-darwin.git"
project_dir = Path("/content/applio-darwin")


def git_environment():
    env = os.environ.copy()
    env["GIT_TERMINAL_PROMPT"] = "0"
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN", "")

    if not token:
        raise RuntimeError(
            "Le dépôt GitHub est privé. Ajoute une seule fois le secret Colab "
            "'GITHUB_TOKEN' dans l'onglet 🔑 Secrets, active l'accès du notebook, "
            "puis relance cette cellule. Le token n'est jamais écrit dans Git."
        )

    askpass = Path("/tmp/applio_git_askpass.sh")
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf '%s\n' "x-access-token" ;;\n'
        '  *Password*) printf '%s\n' "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding="utf-8",
    )
    askpass.chmod(0o700)
    env["GITHUB_TOKEN"] = token
    env["GIT_ASKPASS"] = str(askpass)
    return env


env = git_environment()

if not project_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", GITHUB_REPO, str(project_dir)],
        env=env, check=True
    )
else:
    subprocess.run(
        ["git", "-C", str(project_dir), "remote", "set-url", "origin", GITHUB_REPO],
        env=env, check=True
    )
    subprocess.run(
        ["git", "-C", str(project_dir), "pull", "--ff-only"],
        env=env, check=True
    )

%cd /content/applio-darwin
print("✅ Projet récupéré depuis GitHub.")


## 2. Connecter Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
print("✅ Drive monté.")


## 3. Préparer automatiquement la configuration

In [ ]:
from pathlib import Path
import shutil

example = Path("config/config.example.json")
target = Path("config/config.json")

if not example.exists():
    raise FileNotFoundError(f"Modèle de configuration introuvable : {example}")

if not target.exists():
    shutil.copy2(example, target)
    print("✅ config/config.json créé automatiquement.")
else:
    print("✅ config/config.json déjà présent.")

print(target.read_text(encoding="utf-8"))


## 4. Installer automatiquement Applio

L'installation utilise un environnement Python 3.12 isolé dans le runtime. Rien n'est installé manuellement à chaque nouvelle session.

In [ ]:
!python scripts/setup.py --config config/config.json


## 5. Diagnostic complet

Le diagnostic vérifie l'installation, les ressources, le GPU, le modèle Darwin, l'index et l'audio avant l'inférence.

In [ ]:
import json
from pathlib import Path

with open("config/config.json", encoding="utf-8") as f:
    cfg = json.load(f)

APPLIO_PYTHON = str(Path(cfg.get("runtime", {}).get("python_env_dir", "/content/applio-env")) / "bin" / "python")
print(f"Python Applio : {APPLIO_PYTHON}")


In [ ]:
!{APPLIO_PYTHON} scripts/check_environment.py --config config/config.json


## 6. Inférence — Niveau 1

La conversion utilise automatiquement le modèle `darwin` et son index conservés sur Google Drive, puis sauvegarde le WAV et le log dans `ApplioExported/`.

In [ ]:
!{APPLIO_PYTHON} scripts/inference.py --config config/config.json


In [ ]:
# Écoute directement le résultat ici
import json
from IPython.display import Audio, display

with open("config/config.json", encoding="utf-8") as f:
    cfg = json.load(f)

output_path = f"{cfg['drive']['root']}/{cfg['drive']['export_folder']}/{cfg['model_name']}_output.wav"
if not Path(output_path).exists():
    raise FileNotFoundError(f"Sortie introuvable : {output_path}")
display(Audio(output_path))


---
## Section Entraînement — Niveau 3

⚠️ **Ne pas exécuter avant d'avoir validé les Niveaux 1 et 2.**

Ces cellules sont préparées pour le futur entraînement du modèle `darwin`.

In [ ]:
!{APPLIO_PYTHON} scripts/train.py --config config/config.json --step preprocess


In [ ]:
!{APPLIO_PYTHON} scripts/train.py --config config/config.json --step extract


In [ ]:
!{APPLIO_PYTHON} scripts/train.py --config config/config.json --step index


In [ ]:
!{APPLIO_PYTHON} scripts/train.py --config config/config.json --step train
